In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import explode
from pyspark.sql.functions import expr, lit, to_date

In [0]:
# Configuração de autenticação no ADLS Gen2
spark.conf.set(
    "fs.azure.account.key.nttcase.blob.core.windows.net",
    dbutils.secrets.get(scope="ntt-data", key="storage-key")
)

In [0]:
dbutils.fs.ls(
    "wasbs://bronze@nttcase.blob.core.windows.net/"
)

In [0]:
vaccinations_df = spark.read \
    .option("multiline", "true") \
    .json(
        "wasbs://bronze@nttcase.blob.core.windows.net/vaccinations/ingestion_date=2026-05-12/vaccinations.json"
    )

In [0]:
vaccinations_df.printSchema()

In [0]:
vaccinations_exploded_df = vaccinations_df.select(
    "country",
    "iso_code",
    explode("data").alias("vaccination_data")
)

In [0]:
from pyspark.sql.functions import col

vaccinations_silver_df = vaccinations_exploded_df.select(

    col("country"),

    col("iso_code"),

    col("vaccination_data.date").alias("vaccination_date"),

    col("vaccination_data.daily_people_vaccinated").alias("daily_people_vaccinated"),

    col("vaccination_data.daily_people_vaccinated_per_hundred").alias("daily_people_vaccinated_per_hundred"),

    col("vaccination_data.daily_vaccinations").alias("daily_vaccinations"),

    col("vaccination_data.daily_vaccinations_per_million").alias("daily_vaccinations_per_million"),

    col("vaccination_data.daily_vaccinations_raw").alias("daily_vaccinations_raw"),

    col("vaccination_data.people_fully_vaccinated").alias("people_fully_vaccinated"),

    col("vaccination_data.people_fully_vaccinated_per_hundred").alias("people_fully_vaccinated_per_hundred"),

    col("vaccination_data.people_vaccinated").alias("people_vaccinated"),

    col("vaccination_data.people_vaccinated_per_hundred").alias("people_vaccinated_per_hundred"),

    col("vaccination_data.total_boosters").alias("total_boosters"),

    col("vaccination_data.total_boosters_per_hundred").alias("total_boosters_per_hundred"),

    col("vaccination_data.total_vaccinations").alias("total_vaccinations"),

    col("vaccination_data.total_vaccinations_per_hundred").alias("total_vaccinations_per_hundred")
)

In [0]:
data_ref_carga = "2026-05-12"

vaccinations_silver_df = vaccinations_silver_df.select(

    expr("""
        CASE
            WHEN country IS NULL
                 OR TRIM(LOWER(country)) = 'null'
                 OR TRIM(country) = ''
            THEN 'N/A'
            ELSE CAST(country AS STRING)
        END AS country
    """),

    expr("""
        CASE
            WHEN iso_code IS NULL
                 OR TRIM(LOWER(iso_code)) = 'null'
                 OR TRIM(iso_code) = ''
            THEN 'N/A'
            ELSE CAST(iso_code AS STRING)
        END AS iso_code
    """),

    expr("""
        CASE
            WHEN vaccination_date IS NULL
            THEN DATE('1900-01-01')
            ELSE CAST(vaccination_date AS DATE)
        END AS vaccination_date
    """),

    expr("""
        CASE
            WHEN daily_people_vaccinated IS NULL
            THEN 0
            ELSE CAST(daily_people_vaccinated AS BIGINT)
        END AS daily_people_vaccinated
    """),

    expr("""
        CASE
            WHEN daily_people_vaccinated_per_hundred IS NULL
            THEN 0
            ELSE CAST(daily_people_vaccinated_per_hundred AS DOUBLE)
        END AS daily_people_vaccinated_per_hundred
    """),

    expr("""
        CASE
            WHEN daily_vaccinations IS NULL
            THEN 0
            ELSE CAST(daily_vaccinations AS BIGINT)
        END AS daily_vaccinations
    """),

    expr("""
        CASE
            WHEN daily_vaccinations_per_million IS NULL
            THEN 0
            ELSE CAST(daily_vaccinations_per_million AS BIGINT)
        END AS daily_vaccinations_per_million
    """),

    expr("""
        CASE
            WHEN daily_vaccinations_raw IS NULL
            THEN 0
            ELSE CAST(daily_vaccinations_raw AS BIGINT)
        END AS daily_vaccinations_raw
    """),

    expr("""
        CASE
            WHEN people_fully_vaccinated IS NULL
            THEN 0
            ELSE CAST(people_fully_vaccinated AS BIGINT)
        END AS people_fully_vaccinated
    """),

    expr("""
        CASE
            WHEN people_fully_vaccinated_per_hundred IS NULL
            THEN 0
            ELSE CAST(people_fully_vaccinated_per_hundred AS DOUBLE)
        END AS people_fully_vaccinated_per_hundred
    """),

    expr("""
        CASE
            WHEN people_vaccinated IS NULL
            THEN 0
            ELSE CAST(people_vaccinated AS BIGINT)
        END AS people_vaccinated
    """),

    expr("""
        CASE
            WHEN people_vaccinated_per_hundred IS NULL
            THEN 0
            ELSE CAST(people_vaccinated_per_hundred AS DOUBLE)
        END AS people_vaccinated_per_hundred
    """),

    expr("""
        CASE
            WHEN total_boosters IS NULL
            THEN 0
            ELSE CAST(total_boosters AS BIGINT)
        END AS total_boosters
    """),

    expr("""
        CASE
            WHEN total_boosters_per_hundred IS NULL
            THEN 0
            ELSE CAST(total_boosters_per_hundred AS DOUBLE)
        END AS total_boosters_per_hundred
    """),

    expr("""
        CASE
            WHEN total_vaccinations IS NULL
            THEN 0
            ELSE CAST(total_vaccinations AS BIGINT)
        END AS total_vaccinations
    """),

    expr("""
        CASE
            WHEN total_vaccinations_per_hundred IS NULL
            THEN 0
            ELSE CAST(total_vaccinations_per_hundred AS DOUBLE)
        END AS total_vaccinations_per_hundred
    """)
)

In [0]:
vaccinations_silver_df = vaccinations_silver_df.withColumn(
    "data_ref_carga",
    to_date(lit(data_ref_carga))
)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

vaccinations_silver_df = vaccinations_silver_df.withColumn(
    "id",
    (monotonically_increasing_id() + 1).cast("int")
)

In [0]:
vaccinations_silver_df = vaccinations_silver_df.select(
    "id",
    "country",
    "iso_code",
    "vaccination_date",
    "daily_people_vaccinated",
    "daily_people_vaccinated_per_hundred",
    "daily_vaccinations",
    "daily_vaccinations_per_million",
    "daily_vaccinations_raw",
    "people_fully_vaccinated",
    "people_fully_vaccinated_per_hundred",
    "people_vaccinated",
    "people_vaccinated_per_hundred",
    "total_boosters",
    "total_boosters_per_hundred",
    "total_vaccinations",
    "total_vaccinations_per_hundred",
    "data_ref_carga"
)

In [0]:
display(vaccinations_silver_df)

In [0]:
vaccinations_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("data_ref_carga") \
    .save(
        "wasbs://silver@nttcase.blob.core.windows.net/vaccinations-silver/"
    )

In [0]:
silver_path = "wasbs://silver@nttcase.blob.core.windows.net/vaccinations-silver/"

In [0]:
silver_df = spark.read \
    .format("delta") \
    .load(silver_path)

display(silver_df)

In [0]:
deltaTable = DeltaTable.forPath(
    spark,
    silver_path
)

In [0]:
(
    deltaTable.alias("deltaTable")
    .merge(
        vaccinations_silver_df.alias("source_data"),
        "source_data.id = deltaTable.id"
    )
    .whenMatchedUpdate(set={

        "country": "source_data.country",

        "iso_code": "source_data.iso_code",

        "vaccination_date": "source_data.vaccination_date",

        "daily_people_vaccinated":
            "source_data.daily_people_vaccinated",

        "daily_people_vaccinated_per_hundred":
            "source_data.daily_people_vaccinated_per_hundred",

        "daily_vaccinations":
            "source_data.daily_vaccinations",

        "daily_vaccinations_per_million":
            "source_data.daily_vaccinations_per_million",

        "daily_vaccinations_raw":
            "source_data.daily_vaccinations_raw",

        "people_fully_vaccinated":
            "source_data.people_fully_vaccinated",

        "people_fully_vaccinated_per_hundred":
            "source_data.people_fully_vaccinated_per_hundred",

        "people_vaccinated":
            "source_data.people_vaccinated",

        "people_vaccinated_per_hundred":
            "source_data.people_vaccinated_per_hundred",

        "total_boosters":
            "source_data.total_boosters",

        "total_boosters_per_hundred":
            "source_data.total_boosters_per_hundred",

        "total_vaccinations":
            "source_data.total_vaccinations",

        "total_vaccinations_per_hundred":
            "source_data.total_vaccinations_per_hundred",

        "data_ref_carga":
            "source_data.data_ref_carga"

    })

    .whenNotMatchedInsert(values={

        "id": "source_data.id",

        "country": "source_data.country",

        "iso_code": "source_data.iso_code",

        "vaccination_date": "source_data.vaccination_date",

        "daily_people_vaccinated":
            "source_data.daily_people_vaccinated",

        "daily_people_vaccinated_per_hundred":
            "source_data.daily_people_vaccinated_per_hundred",

        "daily_vaccinations":
            "source_data.daily_vaccinations",

        "daily_vaccinations_per_million":
            "source_data.daily_vaccinations_per_million",

        "daily_vaccinations_raw":
            "source_data.daily_vaccinations_raw",

        "people_fully_vaccinated":
            "source_data.people_fully_vaccinated",

        "people_fully_vaccinated_per_hundred":
            "source_data.people_fully_vaccinated_per_hundred",

        "people_vaccinated":
            "source_data.people_vaccinated",

        "people_vaccinated_per_hundred":
            "source_data.people_vaccinated_per_hundred",

        "total_boosters":
            "source_data.total_boosters",

        "total_boosters_per_hundred":
            "source_data.total_boosters_per_hundred",

        "total_vaccinations":
            "source_data.total_vaccinations",

        "total_vaccinations_per_hundred":
            "source_data.total_vaccinations_per_hundred",

        "data_ref_carga":
            "source_data.data_ref_carga"

    })

    .execute()
)

In [0]:
final_df = spark.read \
    .format("delta") \
    .load(silver_path)

display(final_df)